# AV Embedding Analysis — Sound of Water

Grounded in B08/B09 findings. Produces 3 plots:
- **Plot 1**: Joint PCA + Linear CKA → estimates shared dims (k) between audio/video spaces
- **Plot 2**: Temporal alignment per video (Spearman ρ, audio PC1 ↔ video PC1)
- **Plot 3**: Eigenspectrum + Participation Ratio → diagnoses B09 D-regime

Models: EAT-base (audio, layer 7) + V-JEPA 2.1 ViT-B (video, block 7)

**Setup**: Upload the 6 mp4s from `multimodal_experiments/sound_of_water/videos/` to Colab
or mount your Drive and set `VIDEO_DIR` below.

In [ ]:
# ── Install dependencies ──────────────────────────────────────
!pip install -q transformers==4.51.3 einops timm av
# scikit-learn, pandas, matplotlib, plotly, scipy are pre-installed in Colab

In [ ]:
# ── Mount Drive (optional — use if videos are on Drive) ───────
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ── Config — SET THESE ────────────────────────────────────────
from pathlib import Path

# Option A: videos uploaded directly to Colab
VIDEO_DIR = Path("/content/videos")

# Option B: videos on Drive
# VIDEO_DIR = Path("/content/drive/MyDrive/audio_internship/data/SoundOfWater/videos")

OUT_DIR = Path("/content/av_analysis")
EMB_DIR = OUT_DIR / "embeddings"
OUT_DIR.mkdir(exist_ok=True)
EMB_DIR.mkdir(exist_ok=True)

LAYER_IDX   = 7      # intermediate transformer block (0-based)
AUDIO_SR    = 16_000
AUDIO_CHUNK_MS = 100 # 100ms → 1600 samples
VIDEO_FPS   = 10

In [ ]:
# ── Video metadata (from localisation.csv + containers.yaml) ──
VIDEO_META = {
    "VID_20240116_230040_2.1_16.7.mp4": {
        "label": "V1", "container": "container_1",
        "material": "plastic",     "shape": "cylindrical",
        "setting": "ws-kitchen",   "clean": True,
    },
    "VID_20240118_100817_2.8_23.6.mp4": {
        "label": "V2", "container": "container_2",
        "material": "plastic",     "shape": "cylindrical",
        "setting": "ws-kitchen",   "clean": True,
    },
    "VID_20240118_221233_1.4_24.3.mp4": {
        "label": "V3", "container": "container_10",
        "material": "glass",       "shape": "cylindrical",
        "setting": "ws-kitchen",   "clean": False,
    },
    "VID_20240122_001417_2.3_15.3.mp4": {
        "label": "V4", "container": "container_15",
        "material": "glass",       "shape": "semiconical",
        "setting": "ws-kitchen",   "clean": True,
    },
    "VID_20240131_201458_3.7_24.4.mp4": {
        "label": "V5", "container": "container_18",
        "material": "glass",       "shape": "cylindrical",
        "setting": "ws-room",      "clean": True,  # different setting
    },
    "VID_20240211_204115_2.6_15.0.mp4": {
        "label": "V6", "container": "container_29",
        "material": "plastic",     "shape": "cylindrical",
        "setting": "ws-kitchen",   "clean": False,
    },
    "VID_20240211_204339_3.3_17.0.mp4": {
        "label": "V7", "container": "container_31",
        "material": "plastic_pet", "shape": "semiconical",
        "setting": "ws-kitchen",   "clean": True,
    },
}

COLORS  = {"glass": "#4C8BF5", "plastic": "#F5844C", "plastic_pet": "#9C6FDE"}
MARKERS = {"cylindrical": "o", "semiconical": "^"}

In [ ]:
# ── Imports ───────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=UserWarning)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.dpi": 150,
})

## 1. Load Models

In [ ]:
# ── EAT-base (audio) ──────────────────────────────────────────
from transformers import AutoModel

print("Loading EAT-base...")
eat_model = AutoModel.from_pretrained(
    "worstchan/EAT-base_epoch30_pretrain",
    trust_remote_code=True,
    torch_dtype=torch.float32,
)
eat_model.to(device).eval()
print("EAT loaded ✓")

In [ ]:
# ── V-JEPA 2.1 ViT-B (video) ──────────────────────────────────
print("Loading V-JEPA 2.1 ViT-B...")
hub_model = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_1_vit_base_384',
                            pretrained=False, verbose=False)
vjepa_model = hub_model[0]

ckpt_url = "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt"
ckpt = torch.hub.load_state_dict_from_url(ckpt_url, map_location="cpu", weights_only=False)

if 'encoder' in ckpt:
    sd = {k.replace("module.backbone.", ""): v for k, v in ckpt['encoder'].items()}
    vjepa_model.load_state_dict(sd, strict=False)
else:
    vjepa_model.load_state_dict(ckpt, strict=False)

vjepa_model.to(device).eval()

# Inspect block structure
if hasattr(vjepa_model, 'blocks'):
    vjepa_blocks = vjepa_model.blocks
elif hasattr(vjepa_model, 'encoder') and hasattr(vjepa_model.encoder, 'blocks'):
    vjepa_blocks = vjepa_model.encoder.blocks
else:
    raise RuntimeError("Cannot find 'blocks' on V-JEPA model")

print(f"V-JEPA loaded ✓  |  {len(vjepa_blocks)} transformer blocks")

## 2. Embedding Extraction

In [ ]:
# ── Audio: EAT layer-7 intermediate ───────────────────────────
def load_audio(video_path, target_sr=16_000):
    import torchaudio
    waveform, sr = torchaudio.load(str(video_path))
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    if sr != target_sr:
        waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)
    return waveform.squeeze(0)  # [T]


def extract_eat_layer(video_path, model, layer_idx=7):
    chunk_samples = AUDIO_CHUNK_MS * AUDIO_SR // 1000  # 1600
    waveform = load_audio(video_path)
    core = model.model
    embeddings = []
    n_chunks = len(waveform) // chunk_samples

    for i in tqdm(range(0, len(waveform), chunk_samples), total=n_chunks,
                  desc=f"EAT-L{layer_idx} {Path(video_path).stem[:18]}", leave=False):
        chunk = waveform[i: i + chunk_samples]
        if len(chunk) < chunk_samples:
            continue
        x_in = chunk.float().to(device).view(1, 1, 16, 100)

        with torch.no_grad():
            out = core.local_encoder(x_in)
            x = out[0] if isinstance(out, (tuple, list)) else out

            if hasattr(core, 'fixed_positional_encoder'):
                pad_mask = torch.zeros((x.shape[0], x.shape[1]), dtype=torch.bool, device=device)
                pos_out = core.fixed_positional_encoder(x, padding_mask=pad_mask)
                x = pos_out[0] if isinstance(pos_out, (tuple, list)) else pos_out

            for j in range(layer_idx + 1):
                blk_out = core.blocks[j](x)
                x = blk_out[0] if isinstance(blk_out, (tuple, list)) else blk_out

            embeddings.append(x.mean(1).squeeze(0).cpu().numpy())

    return np.array(embeddings)

In [ ]:
# ── Video: V-JEPA block-7 via forward hook ────────────────────
import torchvision.transforms.functional as TF

def extract_vjepa_layer(video_path, model, blocks, layer_idx=7, fps_target=10):
    """Uses torchvision VideoReader (requires 'av' package)."""
    from torchvision.io import VideoReader

    reader = VideoReader(str(video_path), "video")
    meta = reader.get_metadata()
    orig_fps = meta["video"]["fps"][0] if meta["video"]["fps"] else 30.0
    frame_interval = max(1, round(orig_fps / fps_target))

    IMG_SIZE = 224
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    captured = {}
    def hook_fn(module, inp, out):
        x = out[0] if isinstance(out, (tuple, list)) else out
        captured['feat'] = x.detach()
    handle = blocks[layer_idx].register_forward_hook(hook_fn)

    embeddings, count = [], 0
    try:
        reader.set_current_stream("video")
        pbar = tqdm(desc=f"VJEPA-L{layer_idx} {Path(video_path).stem[:18]}", leave=False)
        for frame in reader:
            if count % frame_interval == 0:
                img = frame["data"].float() / 255.0  # [C, H, W]
                img = TF.resize(img, [IMG_SIZE, IMG_SIZE],
                                interpolation=TF.InterpolationMode.BILINEAR, antialias=True)
                img = (img - mean) / std
                inp = img.unsqueeze(0).unsqueeze(2).to(device)  # [1, C, 1, H, W]
                with torch.no_grad():
                    _ = model(inp)
                feat = captured.get('feat')
                if feat is not None:
                    embeddings.append(feat.mean(1).squeeze(0).cpu().numpy())
                pbar.update(1)
            count += 1
        pbar.close()
    finally:
        handle.remove()
    return np.array(embeddings)

In [ ]:
# ── Extract / load from cache ─────────────────────────────────
def cache_path(vname, modality, layer):
    return EMB_DIR / f"{Path(vname).stem}_{modality}_L{layer}.npy"

audio_embs, video_embs, labels, metas = [], [], [], []

for vname, meta in VIDEO_META.items():
    vpath = VIDEO_DIR / vname
    if not vpath.exists():
        print(f"[SKIP] {vname} not found")
        continue

    a_cp = cache_path(vname, "audio", LAYER_IDX)
    if a_cp.exists():
        a_emb = np.load(a_cp)
        print(f"[cache] audio {meta['label']}: {a_emb.shape}")
    else:
        print(f"[extract] audio {meta['label']}...")
        a_emb = extract_eat_layer(vpath, eat_model, LAYER_IDX)
        np.save(a_cp, a_emb)
        print(f"  → {a_emb.shape}")

    v_cp = cache_path(vname, "video", LAYER_IDX)
    if v_cp.exists():
        v_emb = np.load(v_cp)
        print(f"[cache] video {meta['label']}: {v_emb.shape}")
    else:
        print(f"[extract] video {meta['label']}...")
        v_emb = extract_vjepa_layer(vpath, vjepa_model, vjepa_blocks, LAYER_IDX, VIDEO_FPS)
        np.save(v_cp, v_emb)
        print(f"  → {v_emb.shape}")

    audio_embs.append(a_emb)
    video_embs.append(v_emb)
    labels.append(meta['label'])
    metas.append(meta)

print(f"\n✓ Loaded {len(labels)} videos")

## 3. Metrics

In [ ]:
def participation_ratio(emb):
    """PR = (Σλ)² / Σλ² — effective dimensionality."""
    n_comp = min(emb.shape[0] - 1, emb.shape[1], 100)
    pca = PCA(n_components=n_comp)
    pca.fit(emb)
    lam = pca.explained_variance_
    return float((lam.sum()**2) / (lam**2).sum()), lam


def linear_cka(X, Y):
    """Linear CKA ∈ [0,1]. 0=orthogonal, 1=identical up to linear transform."""
    X = X - X.mean(0); Y = Y - Y.mean(0)
    K = X @ X.T;       L = Y @ Y.T
    num   = np.linalg.norm(K @ L, 'fro')
    denom = np.linalg.norm(K, 'fro') * np.linalg.norm(L, 'fro')
    return float(num / (denom + 1e-10))


def temporal_spearman(a_emb, v_emb):
    n = min(len(a_emb), len(v_emb))
    a_pc1 = PCA(n_components=1).fit_transform(a_emb[:n]).flatten()
    v_pc1 = PCA(n_components=1).fit_transform(v_emb[:n]).flatten()
    rho, _ = spearmanr(a_pc1, v_pc1)
    return float(rho), a_pc1, v_pc1

## Plot 1 — Joint PCA + CKA

In [ ]:
# Align lengths
min_lens  = [min(len(a), len(v)) for a, v in zip(audio_embs, video_embs)]
a_aligned = [a[:n] for a, n in zip(audio_embs, min_lens)]
v_aligned = [v[:n] for v, n in zip(video_embs, min_lens)]
all_a, all_v = np.vstack(a_aligned), np.vstack(v_aligned)

# PCA on audio, project both
pca = PCA(n_components=3).fit(all_a)
a_pca, v_pca = pca.transform(all_a), pca.transform(all_v)
var_exp = pca.explained_variance_ratio_

# CKA
per_vid_cka = [linear_cka(a, v) for a, v in zip(a_aligned, v_aligned)]
global_cka  = linear_cka(all_a, all_v)

cumlen = np.concatenate([[0], np.cumsum(min_lens)])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ax_pc12, ax_pc13, ax_cka = axes

for i, (label, meta) in enumerate(zip(labels, metas)):
    sl   = slice(cumlen[i], cumlen[i+1])
    mark = MARKERS.get(meta['shape'], 'o')
    edge = COLORS.get(meta['material'], 'gray')
    ax_pc12.scatter(a_pca[sl,0], a_pca[sl,1], color='#4EABF5', marker=mark,
                    edgecolors=edge, lw=0.8, alpha=0.5, s=15)
    ax_pc12.scatter(v_pca[sl,0], v_pca[sl,1], color='#F58B4E', marker=mark,
                    edgecolors=edge, lw=0.8, alpha=0.5, s=15)
    ax_pc13.scatter(a_pca[sl,0], a_pca[sl,2], color='#4EABF5', marker=mark,
                    edgecolors=edge, lw=0.8, alpha=0.5, s=15)
    ax_pc13.scatter(v_pca[sl,0], v_pca[sl,2], color='#F58B4E', marker=mark,
                    edgecolors=edge, lw=0.8, alpha=0.5, s=15)
    c = a_pca[sl].mean(0)
    ax_pc12.annotate(label, (c[0], c[1]), fontsize=8, color='#4EABF5', fontweight='bold')

ax_pc12.set(xlabel=f'PC1 ({var_exp[0]*100:.1f}%)', ylabel=f'PC2 ({var_exp[1]*100:.1f}%)',
            title='Audio (blue) vs Video (orange)\nin Audio PCA space')
ax_pc13.set(xlabel=f'PC1 ({var_exp[0]*100:.1f}%)', ylabel=f'PC3 ({var_exp[2]*100:.1f}%)',
            title='PC1 vs PC3')

bar_cols = [COLORS.get(m['material'], 'gray') for m in metas]
ax_cka.bar(labels, per_vid_cka, color=bar_cols, edgecolor='white')
ax_cka.axhline(global_cka, color='black', ls='--', lw=1.5,
               label=f'Global CKA={global_cka:.3f}')
for lbl, val in zip(labels, per_vid_cka):
    ax_cka.text(labels.index(lbl), val + 0.005, f'{val:.3f}', ha='center', fontsize=8)
ax_cka.set(xlabel='Video', ylabel='Linear CKA (audio ↔ video)',
           title=f'CKA per video\n(Global = {global_cka:.3f})')
ax_cka.legend(fontsize=9)

plt.suptitle(f'Plot 1 — Joint AV Embedding Space (EAT-L{LAYER_IDX} + V-JEPA-L{LAYER_IDX})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'plot1_cka_joint_pca.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Global CKA = {global_cka:.3f}')

In [ ]:
# Interactive Plotly version
fig_p = make_subplots(rows=1, cols=2,
                      subplot_titles=['Audio PCA Space (PC1/PC2)', 'CKA per Video'],
                      column_widths=[0.65, 0.35])
sym_map = {'cylindrical': 'circle', 'semiconical': 'triangle-up'}

for i, (label, meta) in enumerate(zip(labels, metas)):
    sl = slice(cumlen[i], cumlen[i+1])
    ts = np.arange(min_lens[i]) * 0.1
    col = COLORS.get(meta['material'], 'gray')
    sym = sym_map.get(meta['shape'], 'circle')
    fig_p.add_trace(go.Scatter(
        x=a_pca[sl,0], y=a_pca[sl,1], mode='markers', name=f'{label} Audio',
        marker=dict(color=ts, colorscale='Blues', size=4, symbol=sym,
                    line=dict(color=col, width=1)),
        text=[f'{label} t={t:.1f}s' for t in ts], hoverinfo='text', legendgroup=label,
    ), row=1, col=1)
    fig_p.add_trace(go.Scatter(
        x=v_pca[sl,0], y=v_pca[sl,1], mode='markers', name=f'{label} Video',
        marker=dict(color=ts, colorscale='Oranges', size=4, symbol=sym,
                    line=dict(color=col, width=1)),
        text=[f'{label} t={t:.1f}s' for t in ts], hoverinfo='text', legendgroup=label,
    ), row=1, col=1)

fig_p.add_trace(go.Bar(
    x=labels, y=per_vid_cka,
    marker_color=[COLORS.get(m['material'], 'gray') for m in metas],
    text=[f'{v:.3f}' for v in per_vid_cka], textposition='outside',
    name='CKA', showlegend=False,
), row=1, col=2)
fig_p.add_hline(y=global_cka, line_dash='dash', line_color='black',
                annotation_text=f'Global CKA={global_cka:.3f}', row=1, col=2)
fig_p.update_layout(height=500, width=1100,
    title=f'Joint AV Embedding Space | Global CKA={global_cka:.3f}')
fig_p.write_html(str(OUT_DIR / 'plot1_cka_joint_pca.html'))
fig_p.show()

## Plot 2 — Temporal Alignment

In [ ]:
rhos, a_trajs, v_trajs = [], [], []
for a, v in zip(audio_embs, video_embs):
    rho, at, vt = temporal_spearman(a, v)
    rhos.append(rho); a_trajs.append(at); v_trajs.append(vt)

best_idx, worst_idx = int(np.argmax(rhos)), int(np.argmin(rhos))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel A: bar chart
ax = axes[0]
ax.bar(labels, rhos, color=[COLORS.get(m['material'], 'gray') for m in metas], edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.axhline(np.mean(rhos), color='black', ls='--', lw=1.2,
           label=f'Mean ρ = {np.mean(rhos):.3f}')
for i, (lbl, rho) in enumerate(zip(labels, rhos)):
    ypos = rho + 0.01 if rho >= 0 else rho - 0.03
    ax.text(i, ypos, f'{rho:.3f}', ha='center', fontsize=8)
    if metas[i]['setting'] == 'ws-room':  # thick border = different setting
        ax.get_children()[i].set_edgecolor('black')
        ax.get_children()[i].set_linewidth(2.5)
ax.set(xlabel='Video', ylabel='Spearman ρ (audio PC1 ↔ video PC1)',
       title='Cross-modal temporal alignment')
ax.legend(fontsize=9)

def norm(x): return (x - x.mean()) / (x.std() + 1e-8)

# Panel B: best
ax2 = axes[1]
t = np.arange(len(a_trajs[best_idx])) * 0.1
ax2.plot(t, norm(a_trajs[best_idx]), color='#4EABF5', lw=1.5, label='Audio PC1')
ax2.plot(t, norm(v_trajs[best_idx]), color='#F58B4E', lw=1.5, label='Video PC1')
ax2.fill_between(t, norm(a_trajs[best_idx]), norm(v_trajs[best_idx]), alpha=0.12, color='gray')
ax2.set(xlabel='Time (s)', ylabel='Normalised PC1',
        title=f"Best: {labels[best_idx]} (ρ={rhos[best_idx]:.3f})\n"
              f"[{metas[best_idx]['material']}, {metas[best_idx]['shape']}, {metas[best_idx]['setting']}]")
ax2.legend(fontsize=9)

# Panel C: worst
ax3 = axes[2]
t2 = np.arange(len(a_trajs[worst_idx])) * 0.1
ax3.plot(t2, norm(a_trajs[worst_idx]), color='#4EABF5', lw=1.5, label='Audio PC1')
ax3.plot(t2, norm(v_trajs[worst_idx]), color='#F58B4E', lw=1.5, label='Video PC1')
ax3.fill_between(t2, norm(a_trajs[worst_idx]), norm(v_trajs[worst_idx]), alpha=0.12, color='gray')
ax3.set(xlabel='Time (s)', ylabel='Normalised PC1',
        title=f"Worst: {labels[worst_idx]} (ρ={rhos[worst_idx]:.3f})\n"
              f"[{metas[worst_idx]['material']}, {metas[worst_idx]['shape']}, {metas[worst_idx]['setting']}]")
ax3.legend(fontsize=9)

plt.suptitle(f'Plot 2 — Cross-Modal Temporal Alignment (EAT-L{LAYER_IDX} ↔ V-JEPA-L{LAYER_IDX})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'plot2_temporal_alignment.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Mean Spearman ρ = {np.mean(rhos):.3f}')

## Plot 3 — Eigenspectrum + Participation Ratio

In [ ]:
pr_audio, pr_video = [], []
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
ax_sa, ax_sv, ax_pr = axes

for i, (label, meta, a_emb, v_emb) in enumerate(zip(labels, metas, audio_embs, video_embs)):
    col = COLORS.get(meta['material'], 'gray')
    ls  = 'dashed' if meta['setting'] == 'ws-room' else 'solid'

    a_pr, a_lam = participation_ratio(a_emb)
    v_pr, v_lam = participation_ratio(v_emb)
    pr_audio.append(a_pr); pr_video.append(v_pr)

    k1 = np.arange(1, len(a_lam)+1)
    ax_sa.plot(k1, np.cumsum(a_lam / a_lam.sum()), color=col, ls=ls, lw=1.5, alpha=0.75, label=label)
    k2 = np.arange(1, len(v_lam)+1)
    ax_sv.plot(k2, np.cumsum(v_lam / v_lam.sum()), color=col, ls=ls, lw=1.5, alpha=0.75, label=label)

for ax, title in [(ax_sa, f'Audio (EAT-L{LAYER_IDX})'), (ax_sv, f'Video (V-JEPA-L{LAYER_IDX})')]:
    ax.axhline(0.8, color='black', ls=':', lw=1, alpha=0.5)
    ax.text(2, 0.82, '80% var', fontsize=8, alpha=0.6)
    ax.set(xlabel='PC rank', ylabel='Cumulative variance explained',
           title=f'{title}\nEigenspectrum', xlim=(1, 50))
    ax.legend(fontsize=8, loc='lower right')

# PR bar chart
x = np.arange(len(labels))
ax_pr.bar(x - 0.175, pr_audio, 0.35, label='Audio PR', color='#4EABF5', edgecolor='white')
ax_pr.bar(x + 0.175, pr_video, 0.35, label='Video PR', color='#F58B4E', edgecolor='white')
for xi, (pa, pv) in enumerate(zip(pr_audio, pr_video)):
    ax_pr.text(xi - 0.175, pa + 0.2, f'{pa:.1f}', ha='center', fontsize=7.5)
    ax_pr.text(xi + 0.175, pv + 0.2, f'{pv:.1f}', ha='center', fontsize=7.5)

# B09 reference lines (D3 = 20 effective dims in 768d → k/d ≈ 2.6%)
ax_pr.axhline(20, color='#888', ls='--', lw=1.2, label='B09-D3 (PR=20, k/d≈2.6%)')
ax_pr.axhline(10, color='#aaa', ls=':', lw=1.2, label='B09-D1/D2 (PR=10, k/d≈1.3%)')
ax_pr.set_xticks(x); ax_pr.set_xticklabels(labels)
ax_pr.set(xlabel='Video', ylabel='Participation Ratio',
          title='Effective dimensionality\n(PR = effective # active dims)')
ax_pr.legend(fontsize=8)

mean_a, mean_v, d = np.mean(pr_audio), np.mean(pr_video), 768
plt.suptitle(f'Plot 3 — Eigenspectrum & PR | Audio mean={mean_a:.1f}/{d} ({100*mean_a/d:.1f}%)  '
             f'Video mean={mean_v:.1f}/{d} ({100*mean_v/d:.1f}%)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'plot3_eigenspectrum_pr.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Mean PR: audio={mean_a:.1f}/{d} ({100*mean_a/d:.1f}%), video={mean_v:.1f}/{d} ({100*mean_v/d:.1f}%)')

## Summary

In [ ]:
metrics = {
    "layer": LAYER_IDX,
    "plot1": {"global_cka": global_cka, "per_video_cka": dict(zip(labels, per_vid_cka))},
    "plot2": {"mean_rho": float(np.mean(rhos)), "per_video_rho": dict(zip(labels, rhos))},
    "plot3": {"mean_pr_audio": mean_a, "mean_pr_video": mean_v,
              "pr_audio_over_d": mean_a/d, "pr_video_over_d": mean_v/d},
}
with open(OUT_DIR / 'metrics_summary.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("=" * 55)
print("TAKE-HOME (B09 context)")
print("=" * 55)
print(f"Global CKA        = {global_cka:.3f}  ({'near-orthogonal' if global_cka < 0.1 else 'weak overlap' if global_cka < 0.3 else 'meaningful overlap'})")
print(f"Mean Spearman ρ   = {np.mean(rhos):.3f}  ({'barely correlated' if abs(np.mean(rhos)) < 0.15 else 'moderate' if abs(np.mean(rhos)) < 0.35 else 'strong'})")
print(f"PR audio/d        = {mean_a:.1f}/{d} = {100*mean_a/d:.1f}%")
print(f"PR video/d        = {mean_v:.1f}/{d} = {100*mean_v/d:.1f}%")
print(f"B09 D3 analogy    = 25% → {0.25*d:.0f}. {'HARDER than D3' if max(mean_a,mean_v) < 0.25*d else 'comparable to D3'}")
print(f"\n✅ Plots saved to {OUT_DIR}")